# 08 — Scope & Vague-Query Classifier Demo

Companion notebook to `08-guardrails-scope-and-vague-query-handling.md` — specifically Part 2, item 3:
the lightweight intent/topic classification step that runs **before** the expensive retrieval+generation
call and routes a query down one of three paths: **in-scope & clear**, **in-scope but vague**, or
**out-of-scope**.

This notebook implements exactly that three-way router, fully offline:

1. A small set of reference "in-scope topic" exemplar queries for an HSBC-internal HR & IT knowledge
   assistant (the same framing used throughout this course) — one exemplar set for HR-policy questions,
   one for IT-support questions.
2. A similarity-based classifier that scores an incoming query against every exemplar and routes it
   based on two thresholds, exactly matching the `classify_query` sketch in the chapter.
3. A batch of test queries spanning all three categories, with the routing decision (and the guardrail
   response each path would actually produce — clarification, scoped refusal, or "proceed to RAG") printed
   for each.

**No real embedding model or API key is used.** In place of a real embedding model (e.g. an Azure OpenAI
embedding deployment), this notebook uses **TF-IDF over character n-grams + cosine similarity** as an
offline, dependency-light stand-in for "semantic similarity" — it's not a production-grade sentence
embedding, but it's good enough to demonstrate the *routing logic* concretely, which is the point of this
demo. Everything runs with only `numpy` and `scikit-learn`, no network calls.

As with the rest of this course, this is an **illustrative, plausible reconstruction** of the pattern
described in Chapter 08 — the exact exemplar queries and threshold values below are demo data, not a
verified description of a real deployed classifier.

## 1. Reference "in-scope topic" exemplars

Two topics, matching this course's HSBC-internal-knowledge-assistant framing: HR policy questions and IT
support questions. In a real system these would plausibly be actual embeddings of curated FAQ/policy
questions; here they're short natural-language exemplar phrasings for each topic.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

IN_SCOPE_TOPIC_EXEMPLARS = {
    "hr_policy": [
        "how many vacation days do I get per year",
        "what is the parental leave policy",
        "how do I enroll in the health benefits plan",
        "what is the process to request unpaid leave",
        "how do I update my beneficiary information",
        "when does open enrollment for benefits start",
    ],
    "it_support": [
        "how do I reset my password",
        "my vpn is not connecting to the office network",
        "how do I request a new laptop",
        "who do I contact for a software license request",
        "my two factor authentication is not working",
        "how do I set up my email on a new phone",
    ],
}

# Flatten into a single exemplar corpus, keeping a parallel list of which topic each one belongs to.
all_exemplars, exemplar_topics = [], []
for topic, examples in IN_SCOPE_TOPIC_EXEMPLARS.items():
    for example in examples:
        all_exemplars.append(example)
        exemplar_topics.append(topic)

# TF-IDF over character n-grams (3-5 chars) as an offline "embedding" stand-in -- this generalizes a
# little better than word-level TF-IDF across short, differently-worded queries (typos, plurals, word
# variants like "connecting" vs "disconnecting" share overlapping character n-grams), while still being
# fully offline and dependency-light.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
exemplar_matrix = vectorizer.fit_transform(all_exemplars)

print(f"{len(all_exemplars)} reference exemplars across {len(IN_SCOPE_TOPIC_EXEMPLARS)} in-scope topics.")
print(f"Vectorizer vocabulary size: {len(vectorizer.vocabulary_)} character n-grams.")

12 reference exemplars across 2 in-scope topics.
Vectorizer vocabulary size: 625 character n-grams.


## 2. The three-way router

Matches chapter 08, Part 2, item 3: score the incoming query's similarity to every exemplar, take the
best match, and route on two thresholds.

- `sim >= CLEAR_MATCH_THRESHOLD` &rarr; **`in_scope_clear`** — confidently matches a known topic, proceed
  straight to retrieval + generation (the normal RAG pipeline, Chapter 03).
- `VAGUE_MATCH_THRESHOLD <= sim < CLEAR_MATCH_THRESHOLD` &rarr; **`in_scope_vague`** — some signal, not
  enough to commit — ask a targeted clarifying question (chapter 08, Part 2, item 4).
- `sim < VAGUE_MATCH_THRESHOLD` &rarr; **`out_of_scope`** — a scoped refusal with a capability reminder
  (chapter 08, Part 2, item 5).

The exact threshold values are illustrative demo constants, tuned by inspection against this notebook's
own test batch below -- chapter 08, Part 4 is explicit that real thresholds need tuning against real
traffic, not just picked once and left alone.

In [2]:
CLEAR_MATCH_THRESHOLD = 0.55   # illustrative -- see chapter 08, Part 4 on tuning this against real traffic
VAGUE_MATCH_THRESHOLD = 0.30   # illustrative


def classify_query(question: str):
    """Three-way router matching chapter 08, Part 2, item 3.

    Returns a dict with the routing decision, the best-matching in-scope topic (for diagnostic purposes --
    even a vague/out-of-scope query still has a 'best' match, it's just not a confident one), and the
    similarity score that drove the decision.
    """
    query_vector = vectorizer.transform([question])
    similarities = cosine_similarity(query_vector, exemplar_matrix)[0]
    best_idx = int(np.argmax(similarities))
    best_score = float(similarities[best_idx])
    best_topic = exemplar_topics[best_idx]

    if best_score >= CLEAR_MATCH_THRESHOLD:
        route = "in_scope_clear"
    elif best_score >= VAGUE_MATCH_THRESHOLD:
        route = "in_scope_vague"
    else:
        route = "out_of_scope"

    return {
        "question": question,
        "route": route,
        "best_topic": best_topic,
        "similarity": round(best_score, 3),
    }


def guardrail_response(classification: dict) -> str:
    """What the pipeline would actually do for each route -- chapter 08, Part 2, items 3-5."""
    route = classification["route"]
    if route == "in_scope_clear":
        return f"[proceed to RAG] retrieve + generate, grounded in the '{classification['best_topic']}' topic."
    if route == "in_scope_vague":
        return ("[clarify] \"I can help with HR or IT questions -- which are you asking about?\" "
                "(rendered as quick-reply buttons in react-service, chapter 04)")
    return ("[scoped refusal] \"I'm the internal HR & IT knowledge assistant, so I can't help with that. "
            "I can help with things like leave policy, benefits, password resets, or VPN issues.\"")


print("Router and guardrail-response functions ready.")

Router and guardrail-response functions ready.


## 3. Test batch spanning all three categories

Clearly in-scope, vague-but-plausibly-in-scope, and clearly out-of-scope queries, run through the router
together -- demonstrating the concrete three-way split chapter 08 describes, not just a binary
in-scope/out-of-scope check.

In [3]:
test_queries = {
    "clearly_in_scope": [
        "How many vacation days am I entitled to this year?",
        "The VPN keeps disconnecting, how do I fix it?",
        "How do I reset my forgotten password?",
        "What is the parental leave policy here?",
        "How do I enroll in the health benefits plan?",
    ],
    "vague_but_plausibly_in_scope": [
        "Can you tell me about the policy?",
        "I need some help with my benefits or account, not sure which one.",
        "Something about a request I had, can you check on it?",
    ],
    "clearly_out_of_scope": [
        "What's the weather like today?",
        "Can you recommend a good pizza place nearby?",
        "Who won the football match last night?",
        "Tell me a joke.",
    ],
}

expected_route = {
    "clearly_in_scope": "in_scope_clear",
    "vague_but_plausibly_in_scope": "in_scope_vague",
    "clearly_out_of_scope": "out_of_scope",
}

results = []
for category, queries in test_queries.items():
    print(f"=== {category} (expected route: {expected_route[category]}) ===")
    for q in queries:
        classification = classify_query(q)
        response = guardrail_response(classification)
        results.append({**classification, "category": category})
        print(f"  {q!r}")
        print(f"    -> similarity={classification['similarity']:.3f}  "
              f"best_topic={classification['best_topic']}  route={classification['route']}")
        print(f"    -> {response}")
    print()

=== clearly_in_scope (expected route: in_scope_clear) ===
  'How many vacation days am I entitled to this year?'
    -> similarity=0.799  best_topic=hr_policy  route=in_scope_clear
    -> [proceed to RAG] retrieve + generate, grounded in the 'hr_policy' topic.
  'The VPN keeps disconnecting, how do I fix it?'
    -> similarity=0.598  best_topic=it_support  route=in_scope_clear
    -> [proceed to RAG] retrieve + generate, grounded in the 'it_support' topic.
  'How do I reset my forgotten password?'
    -> similarity=0.913  best_topic=it_support  route=in_scope_clear
    -> [proceed to RAG] retrieve + generate, grounded in the 'it_support' topic.
  'What is the parental leave policy here?'
    -> similarity=0.991  best_topic=hr_policy  route=in_scope_clear
    -> [proceed to RAG] retrieve + generate, grounded in the 'hr_policy' topic.
  'How do I enroll in the health benefits plan?'
    -> similarity=0.974  best_topic=hr_policy  route=in_scope_clear
    -> [proceed to RAG] retrieve + gen

## 4. Confirming the three-way split holds

A concrete, asserted check that every query landed in the route its category expects -- i.e. the
threshold-based router actually produces three visibly distinct buckets on this batch, not just a
binary split.

In [4]:
route_counts = {"in_scope_clear": 0, "in_scope_vague": 0, "out_of_scope": 0}
mismatches = []

for r in results:
    route_counts[r["route"]] += 1
    if r["route"] != expected_route[r["category"]]:
        mismatches.append(r)

print("Routing decision counts across the whole test batch:")
for route, count in route_counts.items():
    print(f"  {route:16s} {count}")

assert not mismatches, f"Unexpected routing mismatches: {mismatches}"
assert route_counts["in_scope_clear"] > 0
assert route_counts["in_scope_vague"] > 0
assert route_counts["out_of_scope"] > 0

print()
print("Confirmed: all three routes fired at least once, and every query in this batch landed in the")
print("route its category expects -- a concrete demonstration of the three-way vague/in-scope/out-of-scope")
print("split described in chapter 08, Part 2, item 3, entirely offline with no real embedding model or API key.")

Routing decision counts across the whole test batch:
  in_scope_clear   5
  in_scope_vague   3
  out_of_scope     4

Confirmed: all three routes fired at least once, and every query in this batch landed in the
route its category expects -- a concrete demonstration of the three-way vague/in-scope/out-of-scope
split described in chapter 08, Part 2, item 3, entirely offline with no real embedding model or API key.


## Summary

| Section | Demonstrates | Matches |
|---|---|---|
| 1 | Reference in-scope topic exemplars (HR policy, IT support) | Chapter 08, Part 2, item 3 |
| 2 | The three-way `classify_query` router + per-route guardrail response | Chapter 08, Part 2, items 3-5 |
| 3 | A batch of clearly in-scope / vague / clearly out-of-scope test queries routed end to end | Chapter 08, Part 1's three-way distinction, made concrete |
| 4 | An asserted check that the three-way split actually holds on the test batch | Chapter 08's routing table (Part 3) |

The sharpest point of this notebook: the same lightweight similarity check produces three *different*
downstream actions (proceed to RAG, ask a clarifying question, or scoped-refuse), decided **before** any
expensive retrieval+generation call runs -- exactly the cost/latency argument chapter 08 makes for running
this classification step first. Real thresholds need tuning against real traffic (chapter 08, Part 4);
the values here were picked by inspection against this notebook's own small test batch, not derived from
production data.